# Fig1 - Features Distribution Shift (labeled scatter)

Replicates the existing `Fig1_DistributionalShifts.png` (originally an Excel
chart) using the `fig_style.py` journal-figure pipeline shared by the other
figures in this folder. Reads a pooled distributional-shift CSV produced by
`Scripts/fiducial_quality_filtering/distribution_shift.py`
(`shift_pooled_<subset>_<threshold>.csv`) and plots, for the 28 PPG
features: KS test D-statistic (x) vs. IQR-normalized Wasserstein distance
(y), with fixed reference lines marking shift-severity bands.

Change `THRESHOLD` below to read a different cleaning threshold (e.g. 80)
- the script follows the path to whichever `shift_pooled_*_<THRESHOLD>.csv`
exists, it isn't hardcoded to 90.

In [ ]:
import sys
from pathlib import Path

# figures/ -> fig_style.py  |  bp_lgbm/ -> local_paths.py
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path("../").resolve()))

from fig_style import (
    DPI, W_FULL, W_SINGLE, ASPECT, FONT_FAMILY,
    MIN_PX_FULL, MIN_PX_SINGLE,
    apply_base_style, save_fig,
)
from local_paths import DISTRIBUTION_ANALYSIS_DIR, FIGURES_PAPER

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from adjustText import adjust_text

FIG_OUT = FIGURES_PAPER
FIG_OUT.mkdir(parents=True, exist_ok=True)

# ---- Config: which subset/threshold to plot ----
SUBSET = "VitalDB_Train_Subset"   # matches the existing Fig1_DistributionalShifts.png
THRESHOLD = 90                    # change to 80 (or any threshold you've generated) freely

print(f"Output directory: {FIG_OUT.resolve()}")
print(f"Full-page : {W_FULL:.2f} x {W_FULL*ASPECT:.2f} in  ->  {round(W_FULL*DPI)} x {round(W_FULL*ASPECT*DPI)} px @ {DPI} dpi")
print(f"Single-col: {W_SINGLE:.2f} x {W_SINGLE*ASPECT:.2f} in  ->  {round(W_SINGLE*DPI)} x {round(W_SINGLE*ASPECT*DPI)} px @ {DPI} dpi")


## 1 - Load pooled shift results

In [ ]:
# The 28 PPG features, same order/naming used throughout Scripts/bp_lgbm/
# and Scripts/fiducial_quality_filtering/distribution_shift.py.
PPG_FEATURE_NAMES = [
    "IPR", "Tsp", "TWRRF25", "TWRRF50", "Tsw25", "Tsw50", "Tsw75",
    "Tdw25", "Tdw50", "Tdw75", "AUCpi", "IPA", "Av-Au_ratio", "Ab-Aa_ratio",
    "Ac-Aa_ratio", "Ad-Aa_ratio", "Ap2-Ap1_ratio", "AGI", "Kurtosis",
    "Skewness", "L-H_ratio", "ShannonEntropy", "Tpp", "PRV", "FullKurt",
    "FullSkew", "sdPRV", "IQR_PRV",
]

shift_path = DISTRIBUTION_ANALYSIS_DIR / f"shift_pooled_{SUBSET}_{THRESHOLD}.csv"
if not shift_path.exists():
    raise FileNotFoundError(
        f"{shift_path} not found - run distribution_shift.py for this "
        f"subset/threshold first (see Scripts/fiducial_quality_filtering/README.txt)."
    )

shift = pd.read_csv(shift_path).set_index("column")
shift = shift.loc[PPG_FEATURE_NAMES]  # demographics/labels excluded - features only, matching the original figure

print(f"Loaded: {shift_path}")
print(f"{len(shift)} features")
shift[["ks_statistic_D", "normalized_wasserstein"]].head()


## 2 - Build the figure

In [ ]:
# Reference thresholds (fixed shift-severity bands, not derived from data)
KS_YELLOW, KS_RED = 0.10, 0.25
W_GREEN, W_YELLOW, W_RED = 0.20, 0.60, 1.00

# Figure-specific palette
COLOR_POINT  = "#4472C4"  # blue markers
COLOR_LABEL  = "#333333"  # dark gray text, consistent with the other figures
COLOR_GREEN  = "#2ECC71"
COLOR_YELLOW = "#F1C40F"
COLOR_RED    = "#E74C3C"
LEADER_COLOR = "#999999"


def make_shift_fig(width_in: float):
    height_in = width_in * ASPECT
    is_small  = width_in < 5
    point_fs  = 3.5 if is_small else 8
    tick_fs   = 4   if is_small else 8
    axis_fs   = 5   if is_small else 9
    title_fs  = 6   if is_small else 10
    ms        = 3   if is_small else 5

    fig, ax = plt.subplots(figsize=(width_in, height_in), dpi=DPI, layout="constrained")

    # Reference lines first (drawn under the points)
    ax.axvline(KS_YELLOW, color=COLOR_YELLOW, linestyle="--", linewidth=1.1, zorder=1)
    ax.axvline(KS_RED,    color=COLOR_RED,    linestyle="--", linewidth=1.1, zorder=1)
    ax.axhline(W_GREEN,   color=COLOR_GREEN,  linestyle="--", linewidth=1.1, zorder=1)
    ax.axhline(W_YELLOW,  color=COLOR_YELLOW, linestyle="--", linewidth=1.1, zorder=1)
    ax.axhline(W_RED,     color=COLOR_RED,    linestyle="--", linewidth=1.1, zorder=1)

    x = shift["ks_statistic_D"].to_numpy()
    y = shift["normalized_wasserstein"].to_numpy()
    ax.scatter(x, y, s=ms**2, color=COLOR_POINT, zorder=3, edgecolor="none")

    # Auto-avoiding labels with leader lines (adjustText), matching the
    # original Excel chart's data-label + connector-line look.
    # The 4 kurtosis/skewness features sit almost on top of each other in
    # this corner - give their labels a small vertical fan-out as adjustText's
    # starting point so it has room to separate them instead of colliding.
    TIGHT_CLUSTER = {"Skewness": -0.03, "Kurtosis": 0.00, "FullSkew": 0.03, "FullKurt": 0.06}
    texts = [
        ax.text(xi, yi + TIGHT_CLUSTER.get(name, 0.0), name,
                fontsize=point_fs, fontfamily=FONT_FAMILY, color=COLOR_LABEL, zorder=4)
        for xi, yi, name in zip(x, y, shift.index)
    ]
    adjust_text(
        texts, x=x, y=y, ax=ax,
        arrowprops=dict(arrowstyle="-", color=LEADER_COLOR, lw=0.6, shrinkA=0, shrinkB=2),
        expand=(1.6, 2.0),
        force_text=(0.6, 0.9),
        force_explode=(0.5, 0.7),
        max_move=(90, 90),
        iter_lim=4000,
    )

    ax.set_xlabel("KS test (D statistic)", fontsize=axis_fs, fontfamily=FONT_FAMILY)
    ax.set_ylabel("Normalized Wasserstein Distance (IQR units)", fontsize=axis_fs, fontfamily=FONT_FAMILY)
    ax.set_xlim(0, max(0.40, x.max() * 1.1))
    ax.set_ylim(0, max(2.0, y.max() * 1.1))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))

    apply_base_style(ax, grid_axis="both")
    ax.tick_params(axis="both", labelsize=tick_fs)

    ax.set_title(f"Features Distribution Shift ({THRESHOLD} % Threshold)",
                 fontsize=title_fs, fontfamily=FONT_FAMILY, pad=8)

    return fig


fig_full   = make_shift_fig(W_FULL)
fig_single = make_shift_fig(W_SINGLE)

stem = f"Fig1_DistributionalShifts_{SUBSET}_{THRESHOLD}"
save_fig(fig_full,   f"{stem}_full",   FIG_OUT)
save_fig(fig_single, f"{stem}_single", FIG_OUT)

print(f"Full page : {round(W_FULL*DPI)} x {round(W_FULL*ASPECT*DPI)} px")
print(f"Single col: {round(W_SINGLE*DPI)} x {round(W_SINGLE*ASPECT*DPI)} px")

plt.show()


## 3 - Verify output size

In [ ]:
from PIL import Image

for fname, req_w in [
    (f"{stem}_full.png",   MIN_PX_FULL),
    (f"{stem}_single.png", MIN_PX_SINGLE),
]:
    with Image.open(FIG_OUT / fname) as im:
        w, h = im.size
    ok = "OK" if w >= req_w else "FAIL"
    print(f"[{ok}] {fname}: {w} x {h} px  (min required: {req_w} px wide)")
